# TA-03 — Data-Driven Monte Carlo Simulation
## Simulasi Monte Carlo untuk Mengestimasi Risiko Kerugian Bersih Kredit Berdasarkan Tingkat Pemulihan Dana

**Dataset:** Lending Club Loan Data (Kaggle: `ethon0426/lending-club-20072020q1`)

**Tujuan:** mengestimasi distribusi total kerugian bersih portofolio kredit (100 pinjaman per skenario) dan peluang kerugian tersebut melampaui anggaran, berdasarkan data historis pinjaman Lending Club.

**Cara menjalankan**
1. Unduh dataset dari Kaggle, ekstrak, lalu letakkan **file data pinjaman** (CSV, atau CSV terkompresi `.gz`/`.gzip`, Parquet, Excel) di folder yang sama dengan notebook ini, lalu ubah `CSV_PATH` pada sel konfigurasi sesuai nama file. File `LCDataDictionary.xlsx` hanyalah kamus kolom, bukan data.
2. Pilih *Kernel* Python yang sudah memasang `requirements.txt`.
3. Jalankan semua sel dari atas ke bawah (*Run All*). Seluruh angka, grafik, dan kesimpulan dihitung dari file yang Anda muat; tidak ada angka yang ditulis manual.

**Alur:** Import & konfigurasi → Pemuatan data → Pembersihan → Statistik deskriptif → Distribusi & asumsi → Simulasi → Hasil & probabilitas → Histogram → Kesimpulan.

Notebook ini memakai fungsi yang sama persis dengan `app.py` (aplikasi Streamlit). Dengan data, pengaturan, dan seed yang sama, hasilnya identik.

## 1. Import library dan konfigurasi

In [ ]:
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import StrMethodFormatter
from pathlib import Path
from IPython.display import display, Markdown

%matplotlib inline
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
pd.set_option("display.max_columns", 50)

# KONFIGURASI
CSV_PATH = "Loan_status_2007-2020Q3.gzip"   # lokasi file DATA PINJAMAN (CSV / .gz / .gzip / .zip / .parquet / .xlsx), bukan LCDataDictionary.xlsx
SEP = ","                       # pemisah kolom CSV
PERSEN_SAMPEL = 100             # % baris yang dibaca (sampel acak, seed tetap); turunkan bila memori terbatas

# Jika nama kolom tidak terdeteksi otomatis, isi manual. Contoh: {"loan_amnt": "funded_amnt"}
KOLOM_MANUAL = {}

# Jika pengelompokan status otomatis tidak sesuai, isi manual. Contoh: ["Fully Paid"]
STATUS_LUNAS_MANUAL = None
STATUS_GAGAL_MANUAL = None

# Perlakuan recoveries kosong pada pinjaman gagal bayar: "kecualikan" (disarankan) atau "imputasi"
PERLAKUAN_RECOVERIES = "kecualikan"

# Parameter simulasi (default sama dengan aplikasi web)
N_ITER = 10_000          # minimal 5.000
N_PINJAMAN = 100         # jumlah pinjaman per iterasi
SEED = 42                # seed tetap -> hasil dapat direproduksi
ANGGARAN = 250_000.0     # anggaran kerugian (US$); sesuaikan dengan skenario Anda
# =========================================================================

### Fungsi bantu, pembersihan, dan simulasi
Sel berikut berisi definisi fungsi (format angka, pembacaan data per potongan, pembersihan, simulasi bootstrap, ringkasan, histogram). Isinya disalin langsung dari `app.py`.

In [2]:
# Blok ini disalin persis dari app.py (bagian Konstanta s.d. Plot) agar hasil notebook
# dan aplikasi web konsisten.
# --------------------------------------------------------------------------
# Konstanta
# --------------------------------------------------------------------------
KANDIDAT_KOLOM = {
    "loan_amnt": ["loan_amnt", "funded_amnt", "loan_amount", "funded_amount"],
    "recoveries": ["recoveries", "recovery", "total_recoveries"],
    "loan_status": ["loan_status", "status"],
}
LABEL_KOLOM = {
    "loan_amnt": "Kolom jumlah pinjaman",
    "recoveries": "Kolom dana pemulihan",
    "loan_status": "Kolom status pinjaman",
}
UKURAN_CHUNK = 500_000
MIN_ITERASI = 5_000
UKURAN_BATCH = 20_000
SEED_SAMPLING = 42
OPSI_KECUALIKAN = "Kecualikan pinjaman gagal bayar yang recoveries-nya kosong (disarankan)"
OPSI_IMPUTASI = "Imputasi recoveries kosong dengan median rasio pemulihan (recoveries / loan_amnt)"


# --------------------------------------------------------------------------
# Fungsi bantu format (gaya angka Indonesia)
# --------------------------------------------------------------------------
def fmt_usd(x: float) -> str:
    return "US$ " + f"{x:,.0f}".replace(",", ".")


def fmt_int(x: int) -> str:
    return f"{int(x):,}".replace(",", ".")


def fmt_pct(p: float, desimal: int = 2) -> str:
    return f"{p * 100:.{desimal}f}".replace(".", ",") + "%"


# --------------------------------------------------------------------------
# Pembacaan & pembersihan data
# --------------------------------------------------------------------------
_KOMPRESI = {"gzip": "gzip", "zip": "zip"}


def deteksi_format(file) -> str:
    """Deteksi format dari isi file (bukan ekstensi): csv, gzip, zip, parquet, atau xlsx."""
    file.seek(0)
    kepala = file.read(4)
    file.seek(0)
    if kepala[:2] == b"\x1f\x8b":
        return "gzip"
    if kepala == b"PAR1":
        return "parquet"
    if kepala[:2] == b"PK":
        try:
            with zipfile.ZipFile(file) as z:
                nama = z.namelist()
        except zipfile.BadZipFile:
            nama = []
        file.seek(0)
        return "xlsx" if any(n.startswith("xl/") for n in nama) else "zip"
    return "csv"


def periksa_bukan_kamus(kolom) -> None:
    k = {str(c).strip().lower() for c in kolom}
    if {"loanstatnew", "description"} <= k:
        raise ValueError(
            "File ini adalah KAMUS DATA (LCDataDictionary), bukan data pinjaman. Gunakan file data pinjaman "
            "yang berisi kolom loan_amnt, recoveries, dan loan_status."
        )


def baca_pratinjau(file, sep: str, n: int = 100):
    """Baca n baris pertama dari CSV / CSV terkompresi (.gz/.gzip/.zip) / Parquet / Excel."""
    fmt = deteksi_format(file)
    if fmt == "parquet":
        import pyarrow.parquet as pq
        df = next(pq.ParquetFile(file).iter_batches(batch_size=n)).to_pandas()
    elif fmt == "xlsx":
        df = pd.read_excel(file, nrows=n)
    else:
        df = pd.read_csv(file, sep=sep, nrows=n, low_memory=False,
                         compression=_KOMPRESI.get(fmt), encoding_errors="replace")
    file.seek(0)
    periksa_bukan_kamus(df.columns)
    return df, fmt


def iter_potongan(file, sep: str, kolom: list[str]):
    """Hasilkan potongan DataFrame yang hanya berisi kolom yang diperlukan."""
    fmt = deteksi_format(file)
    if fmt == "parquet":
        import pyarrow.parquet as pq
        for batch in pq.ParquetFile(file).iter_batches(batch_size=UKURAN_CHUNK, columns=kolom):
            yield batch.to_pandas()
    elif fmt == "xlsx":
        yield pd.read_excel(file, usecols=kolom)
    else:
        yield from pd.read_csv(file, sep=sep, usecols=kolom, chunksize=UKURAN_CHUNK, low_memory=False,
                               compression=_KOMPRESI.get(fmt), encoding_errors="replace")


def tebak_kolom(daftar_kolom: list[str], kandidat: list[str]) -> str | None:
    peta = {str(k).strip().lower(): k for k in daftar_kolom}
    for c in kandidat:
        if c in peta:
            return peta[c]
    return None


def baca_data(file, sep: str, peta: dict, persen: int, seed: int):
    """Baca hanya 3 kolom yang diperlukan, per potongan (hemat memori).

    Nilai non-numerik pada kolom angka diubah menjadi NaN (BUKAN 0).
    """
    file.seek(0)
    potongan, total, gagal_jumlah, gagal_rec = [], 0, 0, 0
    pembaca = iter_potongan(file, sep, list(peta.values()))
    for i, chunk in enumerate(pembaca):
        total += len(chunk)
        if persen < 100:
            chunk = chunk.sample(frac=persen / 100, random_state=seed + i)
        chunk = chunk.rename(columns={v: k for k, v in peta.items()})
        for kol in ("loan_amnt", "recoveries"):
            sebelum = int(chunk[kol].notna().sum())
            chunk[kol] = pd.to_numeric(chunk[kol], errors="coerce")
            hilang = sebelum - int(chunk[kol].notna().sum())
            if kol == "loan_amnt":
                gagal_jumlah += hilang
            else:
                gagal_rec += hilang
        chunk["loan_status"] = chunk["loan_status"].astype("string").str.strip()
        potongan.append(chunk[["loan_amnt", "recoveries", "loan_status"]])
    if not potongan:
        raise ValueError("File CSV tidak berisi baris data.")
    df = pd.concat(potongan, ignore_index=True)
    df["loan_status"] = df["loan_status"].astype("category")
    info = {
        "baris_file": total,
        "baris_dimuat": len(df),
        "gagal_konversi_jumlah": gagal_jumlah,
        "gagal_konversi_rec": gagal_rec,
    }
    return df, info


def bersihkan(df: pd.DataFrame, status_lunas: list[str], status_gagal: list[str], perlakuan: str):
    """Bangun 'kumpulan pinjaman final' beserta kerugian aktualnya.

    - Lunas        : kerugian = 0 (pinjaman selesai dibayar penuh).
    - Gagal bayar  : kerugian = loan_amnt - recoveries (recoveries harus valid).
    - Lainnya (Current, Late, dst.) : dikecualikan, hasil akhirnya belum diketahui.
    """
    r = {"baris_dianalisis": len(df)}

    jumlah_valid = df["loan_amnt"].notna() & (df["loan_amnt"] > 0)
    status_ada = df["loan_status"].notna()
    r["jumlah_tidak_valid"] = int((~jumlah_valid).sum())
    r["status_kosong"] = int((jumlah_valid & ~status_ada).sum())

    d = df[jumlah_valid & status_ada]
    teks = d["loan_status"].astype(str)
    m_lunas = teks.isin(status_lunas)
    m_gagal = teks.isin(status_gagal) & ~m_lunas
    r["belum_final"] = int((~(m_lunas | m_gagal)).sum())

    lunas = d.loc[m_lunas, ["loan_amnt", "recoveries"]].copy()
    gagal = d.loc[m_gagal, ["loan_amnt", "recoveries"]].copy()

    rec = gagal["recoveries"]
    kosong = rec.isna()
    negatif = rec < 0
    berlebih = rec > gagal["loan_amnt"]
    valid = ~kosong & ~negatif & ~berlebih
    r["gagal_total"] = len(gagal)
    r["rec_kosong"] = int(kosong.sum())
    r["rec_negatif"] = int(negatif.sum())
    r["rec_berlebih"] = int(berlebih.sum())
    r["rec_imputasi"] = 0
    r["rasio_imputasi"] = np.nan

    pakai = valid
    if perlakuan == OPSI_IMPUTASI and valid.any() and kosong.any():
        rasio = float((gagal.loc[valid, "recoveries"] / gagal.loc[valid, "loan_amnt"]).median())
        gagal.loc[kosong, "recoveries"] = rasio * gagal.loc[kosong, "loan_amnt"]
        r["rec_imputasi"] = int(kosong.sum())
        r["rasio_imputasi"] = rasio
        pakai = valid | kosong
    gagal = gagal.loc[pakai].copy()

    gagal["kerugian"] = gagal["loan_amnt"] - gagal["recoveries"]
    gagal["gagal_bayar"] = True
    lunas["kerugian"] = 0.0          # recoveries pada pinjaman lunas tetap NaN (tidak diisi 0)
    lunas["gagal_bayar"] = False
    kumpulan = pd.concat([lunas, gagal], ignore_index=True)

    r["n_lunas"] = len(lunas)
    r["n_gagal"] = len(gagal)
    r["n_kumpulan"] = len(kumpulan)
    r["tingkat_gagal"] = len(gagal) / len(kumpulan) if len(kumpulan) else float("nan")
    return kumpulan, r


# --------------------------------------------------------------------------
# Simulasi Monte Carlo
# --------------------------------------------------------------------------
def simulasi(kerugian: np.ndarray, n_iter: int, n_pinjaman: int, seed: int) -> np.ndarray:
    """Bootstrap: tiap iterasi menarik n_pinjaman pinjaman (dengan pengembalian)
    dari kumpulan pinjaman final historis, lalu menjumlahkan kerugiannya.
    Hasil identik untuk seed, parameter, dan data yang sama."""
    rng = np.random.default_rng(seed)
    hasil = np.empty(n_iter, dtype=float)
    for mulai in range(0, n_iter, UKURAN_BATCH):
        m = min(UKURAN_BATCH, n_iter - mulai)
        idx = rng.integers(0, len(kerugian), size=(m, n_pinjaman))
        hasil[mulai:mulai + m] = kerugian[idx].sum(axis=1)
    return hasil


def ringkas_hasil(hasil: np.ndarray, anggaran: float) -> dict:
    p = float((hasil > anggaran).mean())
    return {
        "rata_rata": float(hasil.mean()),
        "median": float(np.median(hasil)),
        "p95": float(np.percentile(hasil, 95)),
        "std": float(hasil.std(ddof=1)) if len(hasil) > 1 else 0.0,
        "maks": float(hasil.max()),
        "prob": p,
        "se_prob": float(np.sqrt(p * (1 - p) / len(hasil))),
    }


# --------------------------------------------------------------------------
# Plot
# --------------------------------------------------------------------------
def histogram(data, judul, xlabel, warna="#4C78A8", bins=40):
    fig, ax = plt.subplots(figsize=(6, 3.4))
    ax.hist(data, bins=bins, color=warna, edgecolor="white")
    ax.set_title(judul, fontsize=11)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Frekuensi")
    ax.xaxis.set_major_formatter(StrMethodFormatter("{x:,.0f}"))
    ax.yaxis.set_major_formatter(StrMethodFormatter("{x:,.0f}"))
    ax.grid(axis="y", alpha=0.3)
    fig.tight_layout()
    return fig

## 2. Pemuatan dataset dan validasi kolom
Sel berikut memeriksa file, menampilkan pratinjau, lalu memvalidasi tiga kolom wajib: jumlah pinjaman (`loan_amnt`), dana pemulihan (`recoveries`), dan status pinjaman (`loan_status`). Kolom dengan nama setara (mis. `funded_amnt`) ikut dikenali.

In [3]:
if not Path(CSV_PATH).is_file():
    raise FileNotFoundError(
        f"File '{CSV_PATH}' tidak ditemukan. Unduh dataset dari Kaggle "
        "(ethon0426/lending-club-20072020q1), letakkan di folder notebook, atau ubah CSV_PATH."
    )

with open(CSV_PATH, "rb") as f:
    pratinjau, FORMAT_FILE = baca_pratinjau(f, SEP)   # format dideteksi dari isi file
print(f"Format terdeteksi: {FORMAT_FILE} · pratinjau 100 baris pertama · {pratinjau.shape[1]} kolom")
display(pratinjau.head(10))

kolom = list(pratinjau.columns)
PETA = {k: KOLOM_MANUAL.get(k) or tebak_kolom(kolom, kandidat) for k, kandidat in KANDIDAT_KOLOM.items()}

hilang = [LABEL_KOLOM[k] for k, v in PETA.items() if v is None or v not in kolom]
if hilang:
    raise ValueError(f"Kolom berikut tidak ditemukan: {hilang}. Isi KOLOM_MANUAL dengan nama kolom yang benar. "
                     f"Kolom tersedia: {kolom[:40]}...")
if len(set(PETA.values())) < 3:
    raise ValueError("Ketiga peran kolom harus dipetakan ke kolom yang berbeda.")

display(pd.DataFrame({"Peran": [LABEL_KOLOM[k] for k in PETA], "Kolom di dataset": list(PETA.values())}))

FileNotFoundError: File 'lending_club.csv' tidak ditemukan. Unduh dataset dari Kaggle (ethon0426/lending-club-20072020q1), letakkan di folder notebook, atau ubah CSV_PATH.

In [ ]:
with open(CSV_PATH, "rb") as f:
    df_mentah, info_baca = baca_data(f, SEP, PETA, PERSEN_SAMPEL, SEED_SAMPLING)

if df_mentah["loan_amnt"].notna().mean() < 0.5:
    raise ValueError(f"Kolom '{PETA['loan_amnt']}' sebagian besar tidak numerik; periksa pemetaan kolom.")

print(f"Baris di file          : {fmt_int(info_baca['baris_file'])}")
print(f"Baris dimuat           : {fmt_int(info_baca['baris_dimuat'])}")
print(f"Nilai non-numerik -> kosong (bukan 0): loan_amnt {fmt_int(info_baca['gagal_konversi_jumlah'])}, "
      f"recoveries {fmt_int(info_baca['gagal_konversi_rec'])}")
print("\nJumlah nilai kosong per kolom yang dipakai:")
display(df_mentah.isna().sum().rename("kosong").to_frame())

## 3. Pembersihan data dan penanganan nilai kosong

Aturan yang dipakai:
- Baris dengan `loan_amnt` kosong, non-numerik, atau ≤ 0 **dibuang**; baris dengan status kosong **dibuang**.
- **Lunas** (*Fully Paid*): kerugian = 0 (pinjaman selesai dibayar penuh). `recoveries` pada pinjaman ini tidak dipakai dan **tidak diisi 0**.
- **Gagal bayar** (*Charged Off*/*Default*): kerugian = `loan_amnt − recoveries`.
- **Status lain** (*Current*, *Late*, *In Grace Period*, dst.) **dikecualikan**, karena hasil akhirnya belum diketahui. Pinjaman ini tidak boleh dianggap rugi 0.
- `recoveries` **kosong** pada pinjaman gagal bayar **tidak dianggap 0**: dikecualikan (default) atau diimputasi dengan median rasio pemulihan.
- `recoveries` negatif atau lebih besar dari `loan_amnt` dianggap data tidak valid dan dikecualikan.

In [ ]:
hitung = df_mentah["loan_status"].value_counts()
hitung = hitung[hitung > 0]
display(hitung.rename_axis("loan_status").reset_index(name="jumlah"))

opsi = [str(s) for s in hitung.index]
STATUS_LUNAS = STATUS_LUNAS_MANUAL or [s for s in opsi if "fully paid" in s.lower()]
STATUS_GAGAL = STATUS_GAGAL_MANUAL or [s for s in opsi if "charged off" in s.lower() or "default" in s.lower()]

assert STATUS_LUNAS, "Tidak ada status 'lunas' terdeteksi. Isi STATUS_LUNAS_MANUAL."
assert STATUS_GAGAL, "Tidak ada status 'gagal bayar' terdeteksi. Isi STATUS_GAGAL_MANUAL."
assert not (set(STATUS_LUNAS) & set(STATUS_GAGAL)), "Satu status tidak boleh berada di kedua kelompok."

print("Status dianggap LUNAS     :", STATUS_LUNAS)
print("Status dianggap GAGAL BAYAR:", STATUS_GAGAL)
print("Status lain dikecualikan   :", [s for s in opsi if s not in STATUS_LUNAS + STATUS_GAGAL])

In [ ]:
perlakuan = OPSI_IMPUTASI if PERLAKUAN_RECOVERIES == "imputasi" else OPSI_KECUALIKAN
kumpulan, r = bersihkan(df_mentah, STATUS_LUNAS, STATUS_GAGAL, perlakuan)

assert len(kumpulan) > 0, "Tidak ada pinjaman final yang valid. Periksa status dan kolom."

tabel_bersih = [
    ("Baris dianalisis", r["baris_dianalisis"]),
    ("Dibuang: loan_amnt kosong / non-numerik / <= 0", r["jumlah_tidak_valid"]),
    ("Dibuang: status pinjaman kosong", r["status_kosong"]),
    ("Dikecualikan: status belum final (bukan lunas/gagal bayar)", r["belum_final"]),
    ("Pinjaman gagal bayar (sebelum validasi recoveries)", r["gagal_total"]),
    ("   recoveries kosong " + ("(diimputasi)" if r["rec_imputasi"] else "(dikecualikan, BUKAN dianggap 0)"), r["rec_kosong"]),
    ("   dikecualikan: recoveries negatif", r["rec_negatif"]),
    ("   dikecualikan: recoveries > loan_amnt", r["rec_berlebih"]),
    ("Kumpulan final — lunas", r["n_lunas"]),
    ("Kumpulan final — gagal bayar valid", r["n_gagal"]),
    ("Kumpulan final — total", r["n_kumpulan"]),
]
display(pd.DataFrame(tabel_bersih, columns=["Tahap", "Jumlah baris"]).astype({"Jumlah baris": int}))
if r["rec_imputasi"]:
    print(f"Median rasio pemulihan untuk imputasi: {fmt_pct(r['rasio_imputasi'])}")
print(f"Tingkat gagal bayar pada kumpulan final: {fmt_pct(r['tingkat_gagal'])}")

## 4. Statistik deskriptif dan histogram data asli

In [ ]:
gagal = kumpulan[kumpulan["gagal_bayar"]]

tabel = {
    "loan_amnt (semua pinjaman final)": kumpulan["loan_amnt"],
    "recoveries (pinjaman gagal bayar)": gagal["recoveries"],
    "kerugian bersih (pinjaman gagal bayar)": gagal["kerugian"],
    "kerugian bersih (semua pinjaman final)": kumpulan["kerugian"],
}
stat = pd.DataFrame({k: v.describe(percentiles=[0.25, 0.5, 0.75, 0.95]) for k, v in tabel.items()}).T
stat = stat.rename(columns={"count": "n", "mean": "rata-rata", "std": "simpangan baku", "min": "min",
                            "25%": "P25", "50%": "median", "75%": "P75", "95%": "P95", "max": "maks"})
display(stat.round(2))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
for ax, (data, judul, xl, warna) in zip(axes, [
    (kumpulan["loan_amnt"], "Jumlah pinjaman (pinjaman final)", "loan_amnt (US$)", "#4C78A8"),
    (gagal["recoveries"], "Dana pemulihan (gagal bayar)", "recoveries (US$)", "#59A14F"),
    (gagal["kerugian"], "Kerugian bersih (gagal bayar)", "loan_amnt - recoveries (US$)", "#E15759"),
]):
    ax.hist(data, bins=40, color=warna, edgecolor="white")
    ax.set_title(judul); ax.set_xlabel(xl); ax.set_ylabel("Frekuensi")
    ax.xaxis.set_major_formatter(StrMethodFormatter("{x:,.0f}"))
    ax.yaxis.set_major_formatter(StrMethodFormatter("{x:,.0f}"))
    ax.grid(axis="y", alpha=0.3)
fig.suptitle("Histogram data asli", y=1.02)
fig.tight_layout()
plt.show()

## 5. Penentuan distribusi dan asumsi simulasi

Kerugian satu pinjaman dimodelkan sebagai

**L = B × K**, dengan
- **B ~ Bernoulli(p)**: pinjaman gagal bayar (1) atau lunas (0). `p` = tingkat gagal bayar historis pada pinjaman yang hasilnya sudah final.
- **K**: kerugian bersih bila gagal bayar (`loan_amnt − recoveries`), yang mengikuti **distribusi empiris** pinjaman gagal bayar historis. Karena `recoveries` ikut dalam nilai K, ketidakpastian tingkat pemulihan sudah terbawa.

Distribusi K tidak dipaksa mengikuti distribusi parametrik (normal, lognormal, dst.). Sel di bawah memeriksa bentuknya: kerugian per pinjaman gagal bayar umumnya menumpuk pada nilai tertentu dan tidak simetris, sehingga distribusi empiris lebih aman. Menarik pinjaman acak dari kumpulan final (dengan pengembalian) setara dengan **B × K** di atas, dan inilah yang dilakukan simulasi bootstrap.

**Asumsi:** (1) pinjaman saling bebas dan identik secara statistik; (2) data historis yang final mewakili masa depan; (3) pinjaman belum final tidak dimasukkan; (4) satuan kerugian mengikuti dataset (US$).

In [ ]:
gagal = kumpulan[kumpulan["gagal_bayar"]].copy()
gagal["rasio_pemulihan"] = gagal["recoveries"] / gagal["loan_amnt"]

p_gagal = r["tingkat_gagal"]
ringkasan_dist = pd.Series({
    "p (tingkat gagal bayar)": fmt_pct(p_gagal),
    "Rata-rata rasio pemulihan (gagal bayar)": fmt_pct(gagal["rasio_pemulihan"].mean()),
    "Median rasio pemulihan (gagal bayar)": fmt_pct(gagal["rasio_pemulihan"].median()),
    "Gagal bayar dgn recoveries = 0 (nilai tercatat, bukan kosong)": fmt_pct((gagal["recoveries"] == 0).mean()),
    "Rata-rata K (kerugian | gagal bayar)": fmt_usd(gagal["kerugian"].mean()),
    "Kemencengan (skewness) K": f"{gagal['kerugian'].skew():.2f}",
    "Kemencengan (skewness) L (semua pinjaman final)": f"{kumpulan['kerugian'].skew():.2f}",
    "Harapan analitis E[L] per pinjaman": fmt_usd(kumpulan["kerugian"].mean()),
    "p × E[K] (harus sama dengan E[L])": fmt_usd(p_gagal * gagal["kerugian"].mean()),
}, name="nilai")
display(ringkasan_dist.to_frame())
assert np.isclose(p_gagal * gagal["kerugian"].mean(), kumpulan["kerugian"].mean()), "E[L] != p * E[K]"

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(gagal["rasio_pemulihan"].clip(upper=1), bins=40, color="#59A14F", edgecolor="white")
axes[0].set_title("Distribusi rasio pemulihan (recoveries / loan_amnt), pinjaman gagal bayar")
axes[0].set_xlabel("Rasio pemulihan"); axes[0].set_ylabel("Frekuensi"); axes[0].grid(axis="y", alpha=0.3)
x = np.sort(gagal["kerugian"].to_numpy())
axes[1].plot(x, np.arange(1, len(x) + 1) / len(x), color="#E15759")
axes[1].set_title("ECDF kerugian bersih per pinjaman gagal bayar (K)")
axes[1].set_xlabel("Kerugian bersih (US$)"); axes[1].set_ylabel("Probabilitas kumulatif"); axes[1].grid(alpha=0.3)
axes[1].xaxis.set_major_formatter(StrMethodFormatter("{x:,.0f}"))
fig.tight_layout(); plt.show()

## 6. Simulasi Monte Carlo
Setiap iterasi menarik `N_PINJAMAN` pinjaman secara acak (dengan pengembalian) dari kumpulan pinjaman final, lalu menjumlahkan kerugian bersihnya. Dengan `N_ITER` iterasi diperoleh distribusi total kerugian portofolio. Generator acak memakai `SEED` sehingga hasil dapat direproduksi.

In [ ]:
assert N_ITER >= MIN_ITERASI, f"Jumlah iterasi minimal {MIN_ITERASI}."
assert N_PINJAMAN >= 1

kerugian_pool = kumpulan["kerugian"].to_numpy(dtype=float)
hasil = simulasi(kerugian_pool, N_ITER, N_PINJAMAN, SEED)

# Uji reprodusibilitas: seed sama -> hasil identik
ulang = simulasi(kerugian_pool, N_ITER, N_PINJAMAN, SEED)
assert np.array_equal(hasil, ulang), "Simulasi tidak reprodusibel!"
print(f"Simulasi selesai: {fmt_int(N_ITER)} iterasi x {N_PINJAMAN} pinjaman, seed {SEED}. Reprodusibilitas: OK")

# Pemeriksaan konsistensi: rata-rata simulasi vs nilai harapan analitis
harapan = kerugian_pool.mean() * N_PINJAMAN
galat_baku = kerugian_pool.std(ddof=1) * np.sqrt(N_PINJAMAN) / np.sqrt(N_ITER)
selisih = hasil.mean() - harapan
print(f"Nilai harapan analitis : {fmt_usd(harapan)}")
print(f"Rata-rata simulasi     : {fmt_usd(hasil.mean())}")
print(f"Selisih                : {fmt_usd(abs(selisih))}  (galat baku Monte Carlo ≈ {fmt_usd(galat_baku)}; "
      f"{'dalam' if abs(selisih) <= 3 * galat_baku else 'DI LUAR'} 3 galat baku)")

## 7. Kerugian bersih portofolio dan probabilitas melampaui anggaran

In [ ]:
ringkas = ringkas_hasil(hasil, ANGGARAN)

hasil_tabel = pd.Series({
    "Rata-rata kerugian": fmt_usd(ringkas["rata_rata"]),
    "Median kerugian": fmt_usd(ringkas["median"]),
    "Persentil ke-95": fmt_usd(ringkas["p95"]),
    "Simpangan baku": fmt_usd(ringkas["std"]),
    "Kerugian maksimum": fmt_usd(ringkas["maks"]),
    f"Anggaran kerugian": fmt_usd(ANGGARAN),
    "P(total kerugian > anggaran)": fmt_pct(ringkas["prob"]),
    "Galat baku probabilitas (Monte Carlo)": "± " + fmt_pct(ringkas["se_prob"]),
}, name="hasil simulasi")
display(hasil_tabel.to_frame())

# Sensitivitas: probabilitas melampaui anggaran pada beberapa tingkat anggaran
persen_acuan = [50, 75, 90, 95, 99]
sens = pd.DataFrame({
    "Anggaran (US$)": [np.percentile(hasil, q) for q in persen_acuan],
    "Setara persentil": [f"P{q}" for q in persen_acuan],
})
sens["P(kerugian > anggaran)"] = [fmt_pct((hasil > a).mean()) for a in sens["Anggaran (US$)"]]
display(sens)

## 8. Histogram hasil simulasi dengan garis ambang anggaran

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.6))
ax.hist(hasil, bins=60, color="#4C78A8", edgecolor="white")
ax.axvline(ANGGARAN, color="#D62728", linestyle="--", linewidth=2, label=f"Anggaran: {fmt_usd(ANGGARAN)}")
ax.axvline(ringkas["rata_rata"], color="#2CA02C", linewidth=1.5, label=f"Rata-rata: {fmt_usd(ringkas['rata_rata'])}")
ax.axvline(ringkas["p95"], color="#FF7F0E", linestyle=":", linewidth=2, label=f"Persentil 95: {fmt_usd(ringkas['p95'])}")
ax.set_title("Distribusi total kerugian bersih portofolio hasil simulasi")
ax.set_xlabel("Total kerugian bersih per iterasi (US$)"); ax.set_ylabel("Frekuensi")
ax.xaxis.set_major_formatter(StrMethodFormatter("{x:,.0f}"))
ax.grid(axis="y", alpha=0.3); ax.legend()
fig.tight_layout(); plt.show()

In [ ]:
# Simpan hasil simulasi (format sama dengan tombol unduh pada aplikasi web)
keluaran = pd.DataFrame({
    "iterasi": np.arange(1, len(hasil) + 1),
    "total_kerugian_usd": np.round(hasil, 2),
    "melampaui_anggaran": hasil > ANGGARAN,
})
nama_file = f"hasil_simulasi_seed{SEED}.csv"
keluaran.to_csv(nama_file, index=False)
print(f"Tersimpan: {nama_file} ({fmt_int(len(keluaran))} baris)")
display(keluaran.head())

## 9. Kesimpulan
Sel di bawah menyusun kesimpulan **otomatis dari hasil simulasi yang baru saja dijalankan** pada data yang Anda muat, jadi angkanya selalu sesuai dengan output di atas. Klasifikasi risiko (rendah ≤ 5%, sedang 5–20%, tinggi > 20%) hanyalah patokan sederhana untuk membantu membaca hasil. Silakan ubah ambangnya bila tugas Anda memakai kriteria lain.

In [ ]:
def md_usd(x):          # "$" perlu di-escape agar tidak dibaca sebagai rumus oleh Markdown
    return fmt_usd(x).replace("$", r"\$")

s = ringkas
kategori = "rendah" if s["prob"] <= 0.05 else ("sedang" if s["prob"] <= 0.20 else "tinggi")
cukup = "mencukupi" if ANGGARAN >= s["p95"] else "belum mencukupi"
rata_rec = gagal["rasio_pemulihan"].mean()

teks = f"""
### Kesimpulan berdasarkan hasil simulasi

1. **Data dasar.** Dari {fmt_int(r['baris_dianalisis'])} baris yang dianalisis, {fmt_int(r['n_kumpulan'])} pinjaman memiliki hasil akhir yang valid
   ({fmt_int(r['n_lunas'])} lunas dan {fmt_int(r['n_gagal'])} gagal bayar). Tingkat gagal bayar historisnya **{fmt_pct(p_gagal)}**, dengan rata-rata
   rasio pemulihan **{fmt_pct(rata_rec)}** dari jumlah pinjaman. Pinjaman belum final ({fmt_int(r['belum_final'])} baris) tidak dimasukkan.
2. **Kerugian portofolio.** Pada {fmt_int(N_ITER)} iterasi × {N_PINJAMAN} pinjaman (seed {SEED}), rata-rata total kerugian bersih adalah
   **{md_usd(s['rata_rata'])}** (median {md_usd(s['median'])}). Persentil ke-95 sebesar **{md_usd(s['p95'])}**: pada 95% skenario kerugian tidak melebihi angka ini.
3. **Risiko terhadap anggaran.** Dengan anggaran {md_usd(ANGGARAN)}, probabilitas total kerugian melampaui anggaran adalah
   **{fmt_pct(s['prob'])}** (± {fmt_pct(s['se_prob'])} galat Monte Carlo), tergolong risiko **{kategori}** menurut patokan di atas.
   Anggaran ini {cukup} untuk menutup 95% skenario (persentil ke-95 = {md_usd(s['p95'])}).
4. **Sebaran hasil.** Simpangan baku total kerugian {md_usd(s['std'])} dan skenario terburuk {md_usd(s['maks'])}, sehingga hasil terburuk dapat jauh melampaui rata-rata.
5. **Catatan metodologis.** Hasil bersifat reprodusibel (seed tetap) dan sama dengan aplikasi web `app.py` bila data serta parameter sama.
   Kerugian dihitung sebagai `loan_amnt − recoveries` tanpa mengurangi pokok yang sudah dibayar sebelum *charge-off*, sehingga cenderung terlalu besar.
   Pengecualian pinjaman belum final dan asumsi independensi antarpinjaman juga dapat membuat hasil berbeda dari kondisi nyata.
"""
display(Markdown(teks))

## 10. Keterbatasan
1. `loan_amnt − recoveries` merupakan ukuran kerugian kasar. Pembayaran pokok sebelum *charge-off* tidak dikurangkan (kolom seperti `total_rec_prncp` dapat dipakai untuk versi yang lebih akurat).
2. Pinjaman belum final dikecualikan, sehingga tingkat gagal bayar historis dapat bias (*censoring*) terhadap pinjaman yang masih berjalan.
3. Pinjaman dianggap saling bebas dan identik; pengaruh kondisi ekonomi, grade, atau tahun penerbitan tidak dimodelkan.
4. Hasil sepenuhnya bergantung pada data yang dimuat dan pada pengelompokan status yang dipilih.
5. Satuan kerugian mengikuti dataset (US$).